In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv('/content/drive/MyDrive/XAU_1m_data.csv')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data['Ticker'] = 'XAUUSD'

data

,Unnamed: 0,Date,open,high,low,close,volume,Ticker
0,0,2004-06-11 07:18:00,384.00,384.10,384.00,384.00,3,XAUUSD
1,1,2004-06-11 07:23:00,384.10,384.10,384.00,384.00,2,XAUUSD
2,2,2004-06-11 07:24:00,383.80,383.80,383.80,383.80,1,XAUUSD
3,3,2004-06-11 07:25:00,383.80,384.30,383.80,384.30,3,XAUUSD
4,4,2004-06-11 07:27:00,383.80,383.80,383.80,383.80,1,XAUUSD
...,...,...,...,...,...,...,...,...
6559231,6559231,2025-04-25 14:33:00,3299.53,3299.59,3298.34,3298.35,277,XAUUSD
6559232,6559232,2025-04-25 14:34:00,3298.36,3298.88,3298.07,3298.62,327,XAUUSD
6559233,6559233,2025-04-25 14:35:00,3298.61,3298.61,3296.77,3296.81,461,XAUUSD
6559234,6559234,2025-04-25 14:36:00,3296.82,3297.27,3296.40,3296.40,385,XAUUSD


In [4]:
data['Ticker'].unique()

array(['XAUUSD'], dtype=object)

In [5]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2020-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['XAUUSD']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

,Unnamed: 0,Date,open,high,low,close,volume,Ticker
4685138,4685138,2020-01-02 06:00:00,1520.26,1520.30,1520.21,1520.26,15,XAUUSD
4685139,4685139,2020-01-02 06:01:00,1520.26,1520.36,1520.26,1520.30,12,XAUUSD
4685140,4685140,2020-01-02 06:02:00,1520.31,1520.31,1520.21,1520.21,11,XAUUSD
4685141,4685141,2020-01-02 06:03:00,1520.21,1520.26,1520.16,1520.26,14,XAUUSD
4685142,4685142,2020-01-02 06:04:00,1520.26,1520.28,1520.24,1520.26,8,XAUUSD
...,...,...,...,...,...,...,...,...
6559231,6559231,2025-04-25 14:33:00,3299.53,3299.59,3298.34,3298.35,277,XAUUSD
6559232,6559232,2025-04-25 14:34:00,3298.36,3298.88,3298.07,3298.62,327,XAUUSD
6559233,6559233,2025-04-25 14:35:00,3298.61,3298.61,3296.77,3296.81,461,XAUUSD
6559234,6559234,2025-04-25 14:36:00,3296.82,3297.27,3296.40,3296.40,385,XAUUSD


In [6]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('15min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

<ipython-input-6-1792869645>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Date'] = pd.to_datetime(data['Date'])
<ipython-input-6-1792869645>:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)


,Ticker,Date,open,high,low,close,volume
0,XAUUSD,2020-01-02 06:00:00,1520.26,1520.36,1519.90,1519.90,149
1,XAUUSD,2020-01-02 06:15:00,1519.90,1520.08,1519.77,1519.81,159
2,XAUUSD,2020-01-02 06:30:00,1519.81,1520.03,1519.71,1519.85,239
3,XAUUSD,2020-01-02 06:45:00,1519.85,1520.05,1519.39,1519.39,209
4,XAUUSD,2020-01-02 07:00:00,1519.37,1519.41,1518.95,1519.26,276
...,...,...,...,...,...,...,...
125512,XAUUSD,2025-04-25 13:30:00,3304.87,3306.35,3301.22,3303.57,9223
125513,XAUUSD,2025-04-25 13:45:00,3303.54,3303.59,3297.83,3297.98,7832
125514,XAUUSD,2025-04-25 14:00:00,3297.96,3300.93,3296.95,3298.44,6001
125515,XAUUSD,2025-04-25 14:15:00,3298.45,3299.09,3294.35,3298.63,7979


In [7]:
data.dropna()
data

,Ticker,Date,open,high,low,close,volume
0,XAUUSD,2020-01-02 06:00:00,1520.26,1520.36,1519.90,1519.90,149
1,XAUUSD,2020-01-02 06:15:00,1519.90,1520.08,1519.77,1519.81,159
2,XAUUSD,2020-01-02 06:30:00,1519.81,1520.03,1519.71,1519.85,239
3,XAUUSD,2020-01-02 06:45:00,1519.85,1520.05,1519.39,1519.39,209
4,XAUUSD,2020-01-02 07:00:00,1519.37,1519.41,1518.95,1519.26,276
...,...,...,...,...,...,...,...
125512,XAUUSD,2025-04-25 13:30:00,3304.87,3306.35,3301.22,3303.57,9223
125513,XAUUSD,2025-04-25 13:45:00,3303.54,3303.59,3297.83,3297.98,7832
125514,XAUUSD,2025-04-25 14:00:00,3297.96,3300.93,3296.95,3298.44,6001
125515,XAUUSD,2025-04-25 14:15:00,3298.45,3299.09,3294.35,3298.63,7979


In [8]:
data.columns

Index(['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume'], dtype='object')

In [12]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# === Normalized JMA + Signal in One Function ===
def compute_jma_indicator(
    df,
    smooth_length=10,
    norm_period=90,
    upper_thresh=2,
    lower_thresh=-2.5,
    rs_period=10,
    rs_percentile=10
):
    # === JMA Calculation ===
    price_series = df['close']
    alpha = 2.0 / (smooth_length + 1)
    jma = [price_series.iloc[0]]
    for price in price_series.iloc[1:]:
        jma.append(jma[-1] + (price - jma[-1]) * alpha)
    jma_series = pd.Series(jma, index=price_series.index)

    # === Normalization ===
    mean = jma_series.rolling(window=norm_period).mean()
    std = jma_series.rolling(window=norm_period).std()
    normalized_jma = (jma_series - mean) / std

    # === Rogers-Satchell Volatility ===
    # Replace 0s to avoid log(0)
    open_, high, low, close = df['open'].replace(0, np.nan), df['high'], df['low'], df['close'].replace(0, np.nan)
    log_hc = np.log(high / close)
    log_ho = np.log(high / open_)
    log_lc = np.log(low / close)
    log_lo = np.log(low / open_)
    rs_vol = (log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()

    # === RS Volatility Filter ===
    rs_thresh = rs_vol.rolling(window=norm_period).quantile(rs_percentile / 100.0)

    # === Generate Signals ===
    long_condition = (normalized_jma > upper_thresh) & (rs_vol > rs_thresh)
    short_condition = (normalized_jma < lower_thresh) & (rs_vol > rs_thresh)
    signal = pd.Series(0, index=df.index)
    signal[long_condition] = 1
    signal[short_condition] = 2

    # === Output ===
    df['jma'] = jma_series
    df['normalized_jma'] = normalized_jma
    df['rs_vol'] = rs_vol
    df['rs_thresh'] = rs_thresh
    df['Signal'] = signal

    return df

# === Per-Ticker Trade Processor ===
def process_ticker(ticker, ticker_data, initial_capital, fee, slippage, tp_pct, sl_pct):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["Signal"]

        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price = tp_price
                exit_reason = "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price = sl_price
                exit_reason = "Stop Loss"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (fee * entry_price * qty) - (fee * exit_price * qty)
                cumulative_pnl += pnl

                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        if ticker not in open_positions:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

# === Backtest Function ===
def backtest_trades(data, tp_pct, sl_pct):
    initial_capital = 100000
    fee = 0.0004
    slippage = 0.0005

    unique_tickers = data['Ticker'].unique()

    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"), ("Signal", "i4")
    ]

    data_np = np.array(
        list(data[["Ticker", "Date", "close", "open", "high", "low", "Signal"]]
             .itertuples(index=False, name=None)),
        dtype=dtype
    )

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker,
            data_np[data_np['Ticker'] == ticker],
            initial_capital,
            fee,
            slippage,
            tp_pct,
            sl_pct
        ) for ticker in tqdm(unique_tickers)
    )

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

# === Main Execution ===
# Make sure your 'data' DataFrame exists and has columns: Ticker, Date, close, open, high, low
data['Date'] = pd.to_datetime(data['Date'])

# Compute normalized JMA and generate signals
data = data.groupby('Ticker', group_keys=False).apply(compute_jma_indicator)

# Shift Signal for next-bar execution
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(int)

# Run backtest
tradebook = backtest_trades(data, tp_pct=0.1, sl_pct=0.02)

# Save tradebook
tradebook.to_csv("/content/drive/MyDrive/tradebook_Gold.csv", index=False)
tradebook


<ipython-input-12-2712610951>:146: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data = data.groupby('Ticker', group_keys=False).apply(compute_jma_indicator)
100%|██████████| 1/1 [00:00<00:00, 124.94it/s]


,Ticker,Entry Time,Exit Time,Entry Price,Exit Price,PnL,Exit Reason,Cumulative PnL,Direction
0,XAUUSD,2020-01-03 08:15:00,2020-03-09 01:00:00,1542.560895,1695.968576,9861.0220,Take Profit,9861.0220,long
1,XAUUSD,2020-03-09 02:00:00,2020-03-09 06:15:00,1700.970060,1666.117183,-2128.1804,Stop Loss,7732.8416,long
2,XAUUSD,2020-03-10 07:30:00,2020-03-16 12:45:00,1655.421875,1490.624627,9878.9820,Take Profit,17611.8236,short
3,XAUUSD,2020-03-16 14:15:00,2020-03-16 17:00:00,1470.314475,1500.470625,-2131.8204,Stop Loss,15480.0032,short
4,XAUUSD,2020-03-17 17:45:00,2020-03-18 08:30:00,1535.907570,1504.436824,-2128.1804,Stop Loss,13351.8228,long
5,XAUUSD,2020-03-20 08:30:00,2020-04-06 16:45:00,1495.697475,1644.444589,9861.0220,Take Profit,23212.8448,long
6,XAUUSD,2020-04-06 17:00:00,2020-07-08 16:30:00,1649.404290,1813.437547,9861.0220,Take Profit,33073.8668,long
7,XAUUSD,2020-07-08 16:45:00,2020-08-04 18:45:00,1814.626860,1995.091501,9861.0220,Take Profit,42934.8888,long
8,XAUUSD,2020-08-04 19:00:00,2020-08-11 15:45:00,1996.057530,1955.158311,-2128.1804,Stop Loss,40806.7084,long
9,XAUUSD,2020-08-11 16:15:00,2020-08-17 17:15:00,1941.098965,1980.910905,-2131.8204,Stop Loss,38674.8880,short


In [10]:
from itertools import product
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# === Metrics Calculation Function (with R² and Monthly Consistency) ===
def calculate_metrics(tradebook):
    if tradebook.empty:
        return 0, 0, 0, 0

    # Ensure datetime
    tradebook['Date'] = pd.to_datetime(tradebook['Entry Time'])  # Replace with correct column if different

    equity = tradebook['PnL'].cumsum()
    total_profit = equity.iloc[-1]

    roll_max = equity.cummax()
    drawdown = roll_max - equity
    max_drawdown = drawdown.max()

    if max_drawdown == 0:
        ratio = float('inf') if total_profit > 0 else 0
    else:
        ratio = total_profit / max_drawdown

    # R² for equity curve smoothness
    x = np.arange(len(equity)).reshape(-1, 1)
    y = equity.values.reshape(-1, 1)
    model = LinearRegression().fit(x, y)
    r2 = model.score(x, y)

    # Monthly return consistency metric
    tradebook['Month'] = tradebook['Date'].dt.to_period('M')
    monthly_returns = tradebook.groupby('Month')['PnL'].sum()
    if len(monthly_returns) < 2:
        monthly_consistency = 0
    else:
        monthly_mean = monthly_returns.mean()
        monthly_std = monthly_returns.std()
        monthly_consistency = monthly_mean / monthly_std if monthly_std > 0 else float('inf')

    return ratio, total_profit, r2, monthly_consistency

# === Hyperparameter Grid ===
param_grid = {
    'smooth_length': [10, 14],
    'norm_period': [100, 120],
    'upper_thresh': [1.75, 2],
    'lower_thresh': [-2, -2.25, -2.5],
    'rs_period': [10, 14],
    'rs_percentile': [10, 30, 50],
    'tp_pct': [0.05, 0.07, 0.1],
    'sl_pct': [0.01, 0.015, 0.02]
}

# === Generate All Parameter Combinations ===
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
results = []

print(f"Total combinations to test: {len(param_combinations)}\n")

# === Hyperparameter Tuning Loop ===
for i, combo in enumerate(tqdm(param_combinations, desc="Hypertuning"), 1):
    param = dict(zip(keys, combo))

    print(f"\n=== [{i}/{len(param_combinations)}] Testing Parameters ===")
    for k, v in param.items():
        print(f"{k}: {v}")

    temp_data = data.copy()

    def apply_strategy(group):
        return compute_jma_indicator(
            group,
            smooth_length=param['smooth_length'],
            norm_period=param['norm_period'],
            upper_thresh=param['upper_thresh'],
            lower_thresh=param['lower_thresh'],
            rs_period=param['rs_period'],
            rs_percentile=param['rs_percentile']
        )

    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

    # Shift signals to next bar to avoid lookahead bias
    temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
    temp_data = temp_data.dropna(subset=['Signal'])
    temp_data['Signal'] = temp_data['Signal'].astype(int)

    # Run Backtest
    tradebook = backtest_trades(
        temp_data,
        sl_pct=param['sl_pct'],
        tp_pct=param['tp_pct']
    )

    # Evaluate metrics
    ratio, total_profit, r2, monthly_consistency = calculate_metrics(tradebook)

    print(f"➡️ Profit-to-Drawdown Ratio: {ratio:.4f}")
    print(f"➡️ Total Profit: ₹{total_profit:,.2f}")
    print(f"➡️ Equity Curve R²: {r2:.4f}")
    print(f"➡️ Monthly Consistency Score: {monthly_consistency:.4f}")

    result = {
        **param,
        'Profit/Drawdown': ratio,
        'Total Profit': total_profit,
        'Equity R2': r2,
        'Monthly Consistency': monthly_consistency
    }
    results.append(result)

# === Final Results and Export ===
results_df = pd.DataFrame(results)

# Optional: Composite metric for sorting
results_df['Composite Score'] = (
    results_df['Profit/Drawdown'] * 0.5 +
    results_df['Equity R2'] * 0.4 +
    results_df['Monthly Consistency'] * 0.1
)

results_df = results_df.sort_values(by='Composite Score', ascending=False)
results_df.to_csv("/content/drive/MyDrive/hyperparameter_results_adaptive_intraday_full.csv", index=False)

print("\n✅ Hyperparameter tuning complete. Top 5 results:")
print(results_df.head())


Total combinations to test: 3456



Hypertuning:   0%|          | 0/3456 [00:00<?, ?it/s]


=== [1/3456] Testing Parameters ===
smooth_length: 5
norm_period: 80
upper_thresh: 1.75
lower_thresh: -2
rs_period: 10
rs_percentile: 10
tp_pct: 0.03
sl_pct: 0.005


<ipython-input-10-980624002>:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

Hypertuning:   0%|          | 1/3456 [00:01<1:22:32,  1.43s/it]

➡️ Profit-to-Drawdown Ratio: -0.7666
➡️ Total Profit: ₹-59,432.26
➡️ Equity Curve R²: 0.8763
➡️ Monthly Consistency Score: -0.2317

=== [2/3456] Testing Parameters ===
smooth_length: 5
norm_period: 80
upper_thresh: 1.75
lower_thresh: -2
rs_period: 10
rs_percentile: 10
tp_pct: 0.03
sl_pct: 0.01


...

➡️ Profit-to-Drawdown Ratio: 2.0056
➡️ Total Profit: ₹59,534.01
➡️ Equity Curve R²: 0.1320
➡️ Monthly Consistency Score: 0.2567

=== [3455/3456] Testing Parameters ===
smooth_length: 14
norm_period: 90
upper_thresh: 2
lower_thresh: -2.5
rs_period: 14
rs_percentile: 50
tp_pct: 0.1
sl_pct: 0.015


<ipython-input-10-980624002>:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

Hypertuning: 100%|█████████▉| 3455/3456 [1:20:01<00:01,  1.36s/it]

➡️ Profit-to-Drawdown Ratio: 2.7471
➡️ Total Profit: ₹71,638.03
➡️ Equity Curve R²: 0.5058
➡️ Monthly Consistency Score: 0.3000

=== [3456/3456] Testing Parameters ===
smooth_length: 14
norm_period: 90
upper_thresh: 2
lower_thresh: -2.5
rs_period: 14
rs_percentile: 50
tp_pct: 0.1
sl_pct: 0.02


<ipython-input-10-980624002>:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  temp_data = temp_data.groupby('Ticker', group_keys=False).apply(apply_strategy)

Hypertuning: 100%|██████████| 3456/3456 [1:20:02<00:00,  1.39s/it]

➡️ Profit-to-Drawdown Ratio: 1.4564
➡️ Total Profit: ₹41,476.47
➡️ Equity Curve R²: 0.0844
➡️ Monthly Consistency Score: 0.1672

✅ Hyperparameter tuning complete. Top 5 results:
      smooth_length  norm_period  upper_thresh  lower_thresh  rs_period  \
2271             10           90          2.00          -2.5         14   
2223             10           90          2.00          -2.5         10   
1983             10           90          1.75          -2.5         14   
2287             10           90          2.00          -2.5         14   
1935             10           90          1.75          -2.5         10   

      rs_percentile  tp_pct  sl_pct  Profit/Drawdown  Total Profit  Equity R2  \
2271             10     0.1    0.02        10.076177   107292.9628   0.906551   
2223             10     0.1    0.02        10.076177   107292.9628   0.906551   
1983             10     0.1    0.02        10.076177   107292.9628   0.906551   
2287             30     0.1    0.02        10.0